In [ ]:
# === ARRANQUE EN COLAB: árbol de carpetas de la sesión =====================
# Este cuaderno se escribio para correr desde la carpeta `notebook/` de su
# sesión, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese árbol, así que aquí se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/E03_regularizacion_pca"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    _ficha = os.path.join(_RAIZ, "la ficha de la sesiónmd")
    if not os.path.exists(_ficha):
        with open(_ficha, "w", encoding="utf-8") as _fh:
            _fh.write("# marcador de sesión (Colab)\n")
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesión EPE E3 - Demasiadas variables: regularización y PCA

**Curso "Herramientas de Ciencias de Datos" - Modalidad EPE - UPC - Facultad de Negocios**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonatanfigueroagil-creator/Herramientas-de-Ciencias-de-Datos/blob/master/Sesiones_EPE/E03_regularizacion_pca/notebook/EPE_S3_regularizacion_pca.ipynb)

> **Cómo está armada esta sesión.** El contenido avanza en **pasos cortos**: cada paso
> explica **una sola idea** (5-8 minutos de lectura) y de inmediato pide ejecutar y modificar
> **código real** sobre esa misma idea (10-20 minutos). Nunca hay una idea nueva sin su
> práctica inmediata al lado. Cada paso cierra con una celda **"Punto de control - completar:"**
> para registrar la propia conclusión en el momento en que se produce, no al final de la
> sesión. Enfoque EPE: se prioriza la **intuición y la decisión de negocio** sobre el
> formalismo matemático; las fórmulas reales de cada técnica se muestran en su paso, y su
> derivación completa (para quien quiera profundizar) vive en el **Anexo** de
> el cuaderno de la sesión.

## Objetivos de aprendizaje
Al terminar la sesión, el participante es capaz de:
1. Explicar por qué un modelo con **demasiadas variables se sobreajusta** y falla al predecir datos nuevos.
2. Usar la **regularización** como palanca: **RIDGE** contrae los coeficientes hacia cero; **LASSO** los contrae **y selecciona** las variables más importantes.
3. Elegir cuánta regularización aplicar mediante **validación cruzada** (uso práctico, sin derivaciones).
4. Aplicar **PCA** para resumir muchas variables en pocas componentes y **visualizar** clientes o productos en 2D, con lectura de la **varianza explicada**.
5. Decidir **qué técnica** conviene según el objetivo de negocio (seleccionar vs. comprimir).

## Mapa de la sesión (240 minutos reales, en 11 pasos)

| Acto | Minutos | Pasos | Caso de negocio |
|---|---|---|---|
| Apertura | 0-8 | -- | Mapa de la sesión |
| **1. El problema** | 8-30 | Paso 1 | Aseguradora que fija primas por riesgo de zona (Communities and Crime) |
| **2. Contraer** | 30-134 | Pasos 2-6 | La misma aseguradora, con RIDGE y LASSO |
| **3. Comprimir** | 134-203 | Pasos 7-10 | La misma aseguradora, ahora con PCA sobre su cartera (Communities and Crime) |
| **4. Decidir** | 203-240 | Paso 11 + cierre | Recomendación, entregable y control corto |

**Materiales hermanos de esta sesión:** plantillas `plantillas/plantilla_comparacion_modelos.docx` y
`plantillas/guia_componentes_pca.docx`. No hay guía de laboratorio ni drills aparte: la práctica y
las celdas «Punto de control» viven en este mismo cuaderno. El entregable evaluable
(`evaluación/entregable.docx`) aplica PCA a un caso **nuevo** (Wine) que este cuaderno no
cubre, para evaluar si el método se transfiere a datos distintos de los de clase.


In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el núcleo científico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SÍ. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aquí solo se instala lo que Colab NO trae.
import sys

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los ultimos decimales; el método y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "matplotlib": "3.11.1",
    "numpy": "2.5.1",
    "openpyxl": "3.1.5",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "seaborn": "0.13.2",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


In [ ]:
# Librerías de la sesión
import os, sys, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, Lasso, LassoCV
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.exceptions import ConvergenceWarning
# Se silencia SOLO el ruido cosmético (avisos de versión); los avisos de
# CONVERGENCIA (p. ej. LassoCV que no converge) SI se muestran: el docente
# debe poder verlos y no dar por buena una solución no convergida.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("always", category=ConvergenceWarning)

# Estética (paleta de marca UPC: rojo + neutros)
UPC_RED, UPC_INK, UPC_GRAY = "#E4002B", "#2D2D2D", "#9AA0A6"
UPC_BLUE, UPC_GREEN = "#2B5CE4", "#1B9E4B"
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.titleweight": "bold",
                     "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("Librerías OK -", "Colab" if "google.colab" in sys.modules else "entorno local")


In [ ]:
# Rutas robustas (funcionan con nbconvert local y en Colab) y carga de datos.
import importlib.util, hashlib
from urllib.request import Request, urlopen

def _base_dir():
    if "google.colab" in sys.modules:
        return os.getcwd()
    d = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(d, "la ficha de la sesiónmd")):
            return d
        d = os.path.dirname(d)
    d = os.getcwd()
    return os.path.dirname(d) if os.path.basename(d).lower() == "notebook" else d

BASE = _base_dir()
DATA = os.getcwd() if "google.colab" in sys.modules else os.path.join(BASE, "data")
RES = os.path.join(BASE, "resultados"); FIG = os.path.join(BASE, "figuras")
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True); os.makedirs(DATA, exist_ok=True)
XLSX = os.path.join(RES, "E03_resultados.xlsx")

# --- Carga de datos UNIFICADA con data/descargar_datos.py -------------------
# En LOCAL se reutiliza ese módulo (fuente única de nombres, mirrors y checksum).
# En COLAB (sin data/) se usa el respaldo embebido con los MISMOS 128 nombres
# reales, multi-mirror + checksum + verificación de esquema + fallback. NUNCA se
# degradan los nombres a "pred_i": el relato de "variables interpretables" del
# LASSO (PctIlleg, PctKids2Par, racepctblack...) depende de los nombres reales.

# 128 nombres reales de Communities and Crime (idénticos a COMM_COLS de
# data/descargar_datos.py; orden de los @attribute de communities.names).
COMM_COLS = (["state", "county", "community", "communityname", "fold",
    "population", "householdsize", "racepctblack", "racePctWhite",
    "racePctAsian", "racePctHisp", "agePct12t21", "agePct12t29",
    "agePct16t24", "agePct65up", "numbUrban", "pctUrban", "medIncome",
    "pctWWage", "pctWFarmSelf", "pctWInvInc", "pctWSocSec", "pctWPubAsst",
    "pctWRetire", "medFamInc", "perCapInc", "whitePerCap", "blackPerCap",
    "indianPerCap", "AsianPerCap", "OtherPerCap", "HispPerCap",
    "NumUnderPov", "PctPopUnderPov", "PctLess9thGrade", "PctNotHSGrad",
    "PctBSorMore", "PctUnemployed", "PctEmploy", "PctEmplManu",
    "PctEmplProfServ", "PctOccupManu", "PctOccupMgmtProf", "MalePctDivorce",
    "MalePctNevMarr", "FemalePctDiv", "TotalPctDiv", "PersPerFam",
    "PctFam2Par", "PctKids2Par", "PctYoungKids2Par", "PctTeen2Par",
    "PctWorkMomYoungKids", "PctWorkMom", "NumIlleg", "PctIlleg", "NumImmig",
    "PctImmigRecent", "PctImmigRec5", "PctImmigRec8", "PctImmigRec10",
    "PctRecentImmig", "PctRecImmig5", "PctRecImmig8", "PctRecImmig10",
    "PctSpeakEnglOnly", "PctNotSpeakEnglWell", "PctLargHouseFam",
    "PctLargHouseOccup", "PersPerOccupHous", "PersPerOwnOccHous",
    "PersPerRentOccHous", "PctPersOwnOccup", "PctPersDenseHous",
    "PctHousLess3BR", "MedNumBR", "HousVacant", "PctHousOccup",
    "PctHousOwnOcc", "PctVacantBoarded", "PctVacMore6Mos", "MedYrHousBuilt",
    "PctHousNoPhone", "PctWOFullPlumb", "OwnOccLowQuart", "OwnOccMedVal",
    "OwnOccHiQuart", "RentLowQ", "RentMedian", "RentHighQ", "MedRent",
    "MedRentPctHousInc", "MedOwnCostPctInc", "MedOwnCostPctIncNoMtg",
    "NumInShelters", "NumStreet", "PctForeignBorn", "PctBornSameState",
    "PctSameHouse85", "PctSameCity85", "PctSameState85", "LemasSwornFT",
    "LemasSwFTPerPop", "LemasSwFTFieldOps", "LemasSwFTFieldPerPop",
    "LemasTotalReq", "LemasTotReqPerPop", "PolicReqPerOffic", "PolicPerPop",
    "RacialMatchCommPol", "PctPolicWhite", "PctPolicBlack", "PctPolicHisp",
    "PctPolicAsian", "PctPolicMinor", "OfficAssgnDrugUnits",
    "NumKindsDrugsSeiz", "PolicAveOTWorked", "LandArea", "PopDens",
    "PctUsePubTrans", "PolicCars", "PolicOperBudg", "LemasPctPolicOnPatr",
    "LemasGangUnitDeploy", "LemasPctOfficDrugUn", "PolicBudgPerPop",
    "ViolentCrimesPerPop"])

# Checksums SHA256 de los CSV versionados del curso (verifican el mirror del repo).
COMM_SHA256 = "f6d3ff2ca832aafc0d1b7f935d6f39647bd3f3f17436c60ff3009ccf99352460"

_REPO_RAW = ("https://raw.githubusercontent.com/jonatanfigueroagil-creator/"
             "Herramientas-de-Ciencias-de-Datos/master/Sesiones_EPE/"
             "E03_regularizacion_pca/data/")
_UCI_COMM = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
             "communities/communities.data")

def _drive(file_id):
    return f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"

DRIVE_ID = {
    "communities.csv": "1Bs8LCt4rF1x7gvWDLUVt-qmEF0M7MTxF",
}

def _sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def _get(url):
    return urlopen(Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=90).read()

def _descargar_datos_mod():
    cand = os.path.join(BASE, "data", "descargar_datos.py")
    if not os.path.exists(cand):
        return None
    try:
        spec = importlib.util.spec_from_file_location("descargar_datos_e03", cand)
        mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
        return mod
    except Exception as e:
        print("  ! no se pudo importar descargar_datos.py:", e); return None

def _verificar_comm(df):
    for c in ["state", "communityname", "fold", "ViolentCrimesPerPop"]:
        assert c in df.columns, f"communities: falta la columna clave {c}"
    assert df.shape == (1994, 128), f"communities: esquema inesperado {df.shape}"

def _cargar_communities():
    ruta = os.path.join(DATA, "communities.csv")
    mod = _descargar_datos_mod()
    if mod is not None:
        try:
            df = mod.obtener_communities(); _verificar_comm(df); return df
        except Exception as e:
            print("  ! descargar_datos.obtener_communities fallo, uso respaldo:", e)
    if os.path.exists(ruta):
        df = pd.read_csv(ruta, low_memory=False)
        try:
            _verificar_comm(df); return df
        except Exception as e:
            print("  ! communities.csv local no valida, se re-obtiene:", e)
    try:
        print("Descargando communities.csv de la carpeta de datos de Drive...")
        rawb = _get(_drive(DRIVE_ID["communities.csv"]))
        if _sha256_bytes(rawb) != COMM_SHA256:
            print("  ! aviso: checksum de Drive distinto (continuo si el esquema valida)")
        df = pd.read_csv(io.BytesIO(rawb), low_memory=False)
        _verificar_comm(df)
        with open(ruta, "wb") as f: f.write(rawb)
        return df
    except Exception as e:
        print("  ! Drive fallo:", e)
    try:
        print("Descargando communities.csv del mirror del repo del curso...")
        rawb = _get(_REPO_RAW + "communities.csv")
        if _sha256_bytes(rawb) != COMM_SHA256:
            print("  ! aviso: checksum del mirror del repo distinto (continuo si el esquema valida)")
        df = pd.read_csv(io.BytesIO(rawb), low_memory=False)
        _verificar_comm(df)
        with open(ruta, "wb") as f: f.write(rawb)
        return df
    except Exception as e:
        print("  ! mirror del repo fallo:", e)
    print("Descargando Communities and Crime del mirror abierto de UCI...")
    rawb = _get(_UCI_COMM)
    df = pd.read_csv(io.BytesIO(rawb), header=None, names=COMM_COLS,
                     na_values="?", low_memory=False)
    _verificar_comm(df)
    df.to_csv(ruta, index=False, encoding="utf-8")
    return df

comm = _cargar_communities()
print("Communities and Crime:", comm.shape)


In [ ]:
# Primera vista REAL de la tabla (no un resumen ni un conteo): las primeras
# filas tal como quedaron cargadas.
comm.head()


In [ ]:
# Como se interpreta un valor ya normalizado: las columnas predictoras están
# normalizadas a la escala [0, 1] (0 = mínimo observado entre todas las
# comunidades, 1 = máximo). Se describen cuatro columnas representativas.
columnas_ejemplo = ["medIncome", "PctPopUnderPov", "PctUnemployed", "ViolentCrimesPerPop"]
comm[columnas_ejemplo].describe()


### El caso de negocio de esta sesión

**De dónde vienen estos datos.** El archivo `communities.csv` reune 1994 comunidades de
Estados Unidos, con indicadores compilados por Michael Redmond y publicados en el UCI
Machine Learning Repository (id 183, 2009; DOI 10.24432/C53W3X). Combina el censo de
población de 1990, la encuesta policial LEMAS de 1990 y las estadísticas de criminalidad
UCR del FBI de 1995. Cinco columnas (`state`, `county`, `community`, `communityname`,
`fold`) son identificadores, no predictoras, y se descartan del modelo. Varias columnas
tienen datos faltantes (`NaN`).

**Por qué se usa aquí el caso de una aseguradora.** Predecir la tasa de crimen de una
zona a partir de indicadores socioeconómicos es, en la práctica, el mismo problema que
resuelven a diario las áreas de riesgo de **aseguradoras** (para fijar la **prima** de
una póliza de hogar o de negocio según la zona), de entidades financieras (para decidir
la tasa de un crédito hipotecario según la ubicación de la garantía) y de cadenas
retail (para decidir en que ubicaciones abrir tienda). Esta sesión usa el caso de una
**aseguradora de riesgos patrimoniales** que necesita fijar la prima de sus pólizas de
hogar y negocio según el riesgo de cada zona, precisamente porque es la industria donde
este tipo de modelo se usa todos los días para fijar un precio.

**Por qué este caso enseña bien Ridge y LASSO.** Las ~122 columnas predictoras
describen, en muchos casos, aspectos superpuestos de la misma realidad socioeconómica
(varios indicadores de pobreza, varios de estructura familiar, varios de vivienda), por
lo que están fuertemente **correlacionadas** entre sí. Esa es exactamente la situación
para la que Arthur Hoerl y Robert Kennard diseñaron Ridge en 1970: un método que
estabiliza la regresión lineal cuando los predictores son redundantes entre sí.


---
## Acto 1 -- El problema (Min 8-30)
## Paso 1 de 11 -- ¿Por qué muchas variables producen sobreajuste?

*(lectura breve, 8 min)* La aseguradora del caso quiere predecir la tasa de crimen de
una zona (proxy de siniestralidad) a partir de **122 indicadores** socioeconómicos y
policiales. La regresión lineal común (**OLS**), ya trabajada en la sesión anterior, busca los coeficientes que minimizan el
error cuadrático entre lo predicho y lo observado. Lo nuevo aquí es que hay **muchas
variables muy parecidas entre sí** (varios indicadores de pobreza, de estructura
familiar, de vivienda, que en la práctica miden casi lo mismo): a OLS le resulta difícil
decidir cuanto peso (coeficiente) darle a cada una por separado, porque cambios pequeños
en los datos de entrenamiento pueden mover mucho esos pesos.

Cuando un modelo tiene muchas variables puede **sobreajustar** los datos históricos
-incluido su ruido- y ajustarlos casi perfecto; al enfrentar datos **nuevos** falla: es
el **sobreajuste** (*overfitting*). La idea de fondo -sin fórmulas- es el **compromiso
sesgo-varianza**:

| | Modelo demasiado simple | Modelo demasiado complejo (muchas variables) |
|---|---|---|
| Error en datos históricos | Alto | Muy bajo (sobreajusta) |
| Error en datos NUEVOS | Alto | **Alto** (no generaliza) |
| Problema | **Sesgo** alto | **Varianza** alta -> sobreajuste |

> **Regla de oro para juzgar un modelo.** Nunca se juzga por lo bien que ajusta los
> datos con los que se entreno, sino por su **error en datos que no vio** (conjunto de
> prueba). El criterio ya se aplico en E02; aquí se repite con una diferencia: con 122
> variables correlacionadas, ese error de prueba puede ser mucho peor que con pocas.

Las celdas siguientes separan los predictores de la respuesta, dividen los datos en
entrenamiento y prueba, imputan los faltantes por su **promedio** -calculado SOLO con
el entrenamiento, para que ningún valor de prueba entre en ese promedio (dato nuevo
frente a E02, que no tenia NaN; es un paso de preparación, no de modelado)- y ajustan
OLS para comparar su R2 (ya definido en E02, sección 3.2) en entrenamiento contra
prueba.

In [ ]:
# Columnas que NO son predictoras (identificadores) más la columna objetivo:
# se separan del resto antes de ajustar cualquier modelo.
no_predictoras = ["state", "county", "community", "communityname", "fold"]
no_predictoras = [c for c in no_predictoras if c in comm.columns]

y = comm["ViolentCrimesPerPop"].astype(float)          # lo que se predice
X = comm.drop(columns=no_predictoras + ["ViolentCrimesPerPop"]).apply(pd.to_numeric, errors="coerce")
print("Predictores disponibles:", X.shape[1], "| columnas con algun faltante:", int((X.isna().sum() > 0).sum()))


In [ ]:
# Conjunto de prueba (30% de las filas, mismo criterio de E02, sección 3.4): el
# split se hace ANTES de imputar los NaN, para que ningún valor de prueba entre
# en el promedio de imputación (la celda siguiente imputa SOLO con el promedio
# del entrenamiento; el mismo principio que, dos celdas más abajo, hace que el
# escalador también se ajuste solo con el entrenamiento). random_state fija la
# semilla para que el resultado no cambie al repetir.
Xtr_df, Xte_df, ytr, yte = train_test_split(X, y.values, test_size=0.30, random_state=42)
print("Entrenamiento:", Xtr_df.shape, " Prueba:", Xte_df.shape)

In [ ]:
# Varias columnas tienen NaN (datos faltantes): se imputan por el promedio de
# cada columna, calculado SOLO con las filas de ENTRENAMIENTO (paso de
# preparación, no de modelado) y aplicado igual a ambos conjuntos. Si el
# promedio se calculara con las 1994 filas completas, el valor de imputación
# de cada fila de prueba dependeria en parte de las demás filas de prueba: es
# fuga de información (data leakage) hacia el conjunto que se usa para juzgar
# al modelo.
medias_train = Xtr_df.mean()
Xtr = Xtr_df.fillna(medias_train).values
Xte = Xte_df.fillna(medias_train).values
print("NaN restantes:", int(pd.DataFrame(Xtr).isna().sum().sum() + pd.DataFrame(Xte).isna().sum().sum()))

In [ ]:
# Estandarizar: cada columna queda con promedio 0 y desviación estándar 1.
# El escalador se AJUSTA solo con el entrenamiento (para no "mirar" el test);
# más adelante (Paso 2) se explica por qué este paso es obligatorio para
# regularizar.
escalador = StandardScaler().fit(Xtr)
Ztr = escalador.transform(Xtr)
Zte = escalador.transform(Xte)
print("Datos estandarizados:", Ztr.shape)


In [ ]:
# OLS (ya visto en E02, sección 3.1), sin ninguna penalización.
ols = LinearRegression().fit(Ztr, ytr)

r2_train_ols = r2_score(ytr, ols.predict(Ztr))
r2_test_ols = r2_score(yte, ols.predict(Zte))
print(f"OLS - R2 en entrenamiento: {r2_train_ols:.3f}")
print(f"OLS - R2 en prueba:        {r2_test_ols:.3f}")
print(f"Brecha (entrenamiento - prueba): {r2_train_ols - r2_test_ols:.3f}")


📖 **Lectura de negocio -- que dice la salida.** El OLS con las 122 variables ajusta
bien el entrenamiento (R2 ~0.71) y también predice de forma razonable en **prueba**
(R2 ~0.62): no colapsa. Pero la brecha entre ambos (~0.09) es la firma del
**sobreajuste**: hay un margen de mejora que los Pasos 2-6 recuperan, sobre todo con
LASSO.

⚠️ **Riesgo: la causa no es solo el número de variables.** Hay **~1400 filas de
entrenamiento para 122 variables** (razón 11:1): sobran filas, no es el caso p≈n. Buena
parte de esa brecha viene de que esos 122 indicadores socioeconómicos están **muy
correlacionados entre sí** (multicolinealidad): con columnas casi redundantes, el
ajuste de mínimos cuadrados reparte el peso entre ellas de forma inestable, lo que
infla la varianza de los coeficientes frente a datos nuevos. Los Pasos 2-6 muestran la
corrección.

💡 **Para la aseguradora.** Un modelo que mejora algo el ajuste en el histórico pero
pierde más precisión en zonas nuevas que RIDGE o LASSO fija primas menos confiables. El
único criterio válido es el error en datos que el modelo no vio.

**Punto de control -- completar:** En una línea, ¿por qué un R2 de entrenamiento alto no alcanza para confiar en un modelo con 122 variables?

> _______________________________________________________________________


---
## Acto 2 -- Contraer: RIDGE y LASSO (Min 30-134)
## Paso 2 de 11 -- ¿Qué hace RIDGE?

*(lectura breve, 7 min)* Ridge ajusta los mismos coeficientes que OLS, pero cambia la
meta: en vez de minimizar solo el error de predicción, minimiza el error **más una
penalización** que crece con el tamaño de los coeficientes (la suma de los coeficientes
al cuadrado). Con $y_i$ el valor real, $\hat{y}_i$ el valor que predice el modelo y
$\beta_j$ cada coeficiente:

$$\min_{\beta}\ \sum_i (y_i-\hat{y}_i)^2 + \alpha\sum_j \beta_j^2 \qquad\text{(RIDGE, penalización L2)}$$

El primer termino es exactamente lo que ya minimiza OLS; el segundo es la penalización
nueva: `alpha` (la misma letra que usa `scikit-learn`; en la literatura se la llama
$\lambda$) controla cuanto pesa esa penalización frente al error. A mayor `alpha`, mayor
penalización y coeficientes más pequeños; en `alpha = 0`, Ridge es idéntico a OLS. Por
ahora se fija `alpha` de forma manual en un valor cualquiera, solo para ver el efecto de contracción (*shrinkage*)
sobre los coeficientes; el Paso 3 muestra cómo elegirlo con criterio.

Es la respuesta que Hoerl y Kennard diseñaron en 1970 justamente para la
**multicolinealidad** detectada en el Paso 1: reparte el peso entre variables
correlacionadas en vez de dejar que el ajuste lo haga de forma errática.


In [ ]:
# Ridge con un alpha elegido a mano (arbitrario, solo para ilustrar el efecto
# de la penalización). El Paso 3 muestra como elegirlo bien.
ridge_manual = Ridge(alpha=50).fit(Ztr, ytr)
print("Ridge (alpha=50) ajustado.")


In [ ]:
# Comparar el tamaño de los coeficientes: Ridge los "encoge" frente a OLS.
print("Tamaño promedio de los coeficientes (valor absoluto):")
print(f"  OLS:              {np.abs(ols.coef_).mean():.4f}")
print(f"  Ridge (alpha=50): {np.abs(ridge_manual.coef_).mean():.4f}")
print()
print("Coeficientes exactamente en cero:")
print(f"  OLS:              {int(np.sum(np.abs(ols.coef_) <= 1e-8))}")
print(f"  Ridge (alpha=50): {int(np.sum(np.abs(ridge_manual.coef_) <= 1e-8))}")


📖 **Lectura de negocio -- que dice la salida.** Con la penalización activa, los
coeficientes de Ridge son, en promedio, más pequeños que los de OLS: es el efecto de
contracción (*shrinkage*) que caracteriza a la regularización. Y ninguno llega a cero: Ridge **conserva
las 122 variables**, solo las modera.

💡 **Para la aseguradora.** Ridge no decide que indicador dejar de recolectar (eso lo
hace LASSO, Paso 4); estabiliza el modelo para que la prima no dependa de un reparto de
coeficientes casi al azar entre indicadores redundantes.


**Punto de control -- completar:** Si `alpha` sube mucho más (por ejemplo, a 5000), ¿qué le pasaría al tamaño de los coeficientes, y por qué ese extremo tampoco conviene?

> _______________________________________________________________________


## Paso 3 de 11 -- ¿Cómo se elige la fuerza de la penalización (alpha) por validación cruzada?

*(lectura breve, 6 min)* El `alpha=50` del Paso 2 se fijó de forma manual, sin ningún criterio:
podría ser demasiado pequeño (y no corregir casi nada) o demasiado grande (y aplanar tanto
los coeficientes que el modelo deje de usar la información real). El criterio correcto
es la **validación cruzada** (*cross-validation*, CV): se prueban muchos valores de
`alpha` y, para cada uno, se mide que tan bien predice en partes del entrenamiento
apartadas por turnos, sin tocar nunca el conjunto de prueba. Se elige el `alpha` que da
el mejor resultado promedio en esa rotación. En `scikit-learn`, `RidgeCV` automatiza
todo ese barrido.


In [ ]:
# np.logspace(-3, 3, 50) genera 50 valores de alpha repartidos en escala
# logaritmica entre 10^-3 = 0.001 y 10^3 = 1000: cubre desde una penalización
# casi nula hasta una muy fuerte.
rejilla_alphas = np.logspace(-3, 3, 50)
print("Valores de alpha probados:", len(rejilla_alphas), "entre", rejilla_alphas[0], "y", rejilla_alphas[-1])


In [ ]:
# RidgeCV prueba cada alpha de la rejilla con validación cruzada (usa el
# entrenamiento en rotación) y se queda con el que mejor generaliza, sin haber
# mirado nunca el conjunto de prueba.
ridge_cv = RidgeCV(alphas=rejilla_alphas).fit(Ztr, ytr)
print(f"alpha elegido por validación cruzada: {ridge_cv.alpha_:.3f}")


📖 **Lectura de negocio.** Nadie tecleo el `alpha` final: lo eligió la validación
cruzada, prueba 50 valores y se queda con el que mejor predice en datos que no
participaron de su propio ajuste.

⚠️ **Riesgo: elegir `alpha` por el ajuste de entrenamiento.** Si se eligiera el `alpha`
que mejor ajusta el **entrenamiento**, la respuesta seria siempre `alpha = 0` (o casi):
el sobreajuste del Paso 1 volvería a aparecer. La validación cruzada nunca mira el
ajuste de entrenamiento para decidir `alpha`.


**Punto de control -- completar:** ¿Qué pasaría si se decidiera el `alpha` por el valor que da el menor error en el conjunto de ENTRENAMIENTO en vez de por validación cruzada?

> _______________________________________________________________________


## Paso 4 de 11 -- ¿Qué hace LASSO, y en qué se diferencia de RIDGE?

*(lectura breve, 8 min)* LASSO cambia la forma de la penalización: en vez de sumar los
coeficientes **al cuadrado** (L2, Ridge), suma sus **valores absolutos** (L1):

$$\min_{\beta}\ \sum_i (y_i-\hat{y}_i)^2 + \alpha\sum_j |\beta_j| \qquad\text{(LASSO, penalización L1)}$$

Esa diferencia de forma tiene una consecuencia práctica considerable: mientras Ridge contrae
todos los coeficientes sin anular ninguno (Paso 2), LASSO aplica un **umbral suave**
(*soft-threshold*) que **lleva a cero** los coeficientes pequeños. Con variables ya
estandarizadas (Paso 1) y aproximadamente incorreladas, el efecto de cada penalización
sobre un coeficiente $z$ (su valor según OLS) se ve en una línea:

$$\hat{\beta}_j^{\text{ridge}} = \frac{z_j}{1+\alpha} \qquad\qquad \hat{\beta}_j^{\text{lasso}} = \operatorname{sign}(z_j)\,\max(|z_j|-\alpha,\,0)$$

Números para fijar la idea (con $\alpha = 0.20$): con $z = 0.50$, Ridge da
$0.50/1.20 = \mathbf{0.42}$ y LASSO da $\max(0.50-0.20, 0) = \mathbf{0.30}$; con
$z = 0.15$, Ridge da $0.15/1.20 = \mathbf{0.125}$ (se conserva) pero LASSO da
$\max(0.15-0.20, 0) = \mathbf{0}$ (variable **eliminada**). Ese cero es la diferencia
entre contraer (Ridge) y seleccionar (LASSO): LASSO hace **selección automática de
variables** ("de 122 indicadores, conservar solo los que realmente importan"), un modelo
ralo, de bajo costo operativo y auditable.

Por ahora se fija `alpha` de forma manual, solo para ver el efecto; el Paso 5 muestra la
versión por validación cruzada y compara los tres modelos.


In [ ]:
# LASSO con un alpha elegido a mano (arbitrario, solo para ilustrar el efecto
# de selección). El Paso 5 muestra como elegirlo bien.
lasso_manual = Lasso(alpha=0.01, max_iter=100000).fit(Ztr, ytr)
print("LASSO (alpha=0.01) ajustado.")


In [ ]:
# Cuantas variables sobreviven (coeficiente distinto de cero) en cada modelo.
n_ols = int(np.sum(np.abs(ols.coef_) > 1e-8))
n_ridge = int(np.sum(np.abs(ridge_manual.coef_) > 1e-8))
n_lasso = int(np.sum(np.abs(lasso_manual.coef_) > 1e-8))
print(f"Variables con coeficiente != 0 -> OLS: {n_ols} | Ridge: {n_ridge} | LASSO: {n_lasso}")


📖 **Lectura de negocio.** OLS y Ridge conservan las 122 variables (Ridge las
contrae, no las elimina); LASSO ya, con este `alpha` de prueba, deja varias en cero
exacto. Es la **selección automática de variables**.

⚠️ **Riesgo: correlación inestable.** Cuando dos indicadores están muy correlacionados,
LASSO tiende a quedarse con **uno del grupo casi al azar** y descartar el otro, aunque
ambos aporten señal parecida; el Paso 6 retoma esto con los tres avisos honestos sobre
que significa "sobrevivir" al LASSO.


**Punto de control -- completar:** Con la fórmula del umbral suave, si $z = 0.30$ y $\alpha = 0.20$, ¿cuánto le queda al coeficiente en LASSO? ¿Y en RIDGE (con el mismo $\alpha$)?


## Paso 5 de 11 -- ¿Cómo se elige el alpha del LASSO, y cuál de los tres modelos conviene?

*(lectura breve, 6 min)* Igual que con Ridge (Paso 3), el `alpha` del LASSO se elige
por **validación cruzada** (`LassoCV`), nunca de forma arbitraria ni por el ajuste de entrenamiento.
Con los tres modelos ya ajustados con su mejor `alpha` (OLS no tiene `alpha`; Ridge y
LASSO lo eligen por CV), corresponde la comparación que responde la pregunta del Paso 1: cual
conviene, y por qué.


In [ ]:
# LassoCV prueba muchos valores de alpha con validación cruzada (cv=5 folds)
# y elige el que mejor generaliza.
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=100000).fit(Ztr, ytr)
print(f"alpha elegido por validación cruzada: {lasso_cv.alpha_:.5f}")


In [ ]:
# Tabla comparativa final: MSE de prueba, R2 de prueba y número de variables
# de los tres modelos, todos evaluados en el mismo conjunto de PRUEBA.
filas = []
for nombre, m in [("OLS", ols), ("RIDGE", ridge_cv), ("LASSO", lasso_cv)]:
    pred = m.predict(Zte)
    n_vars = int(np.sum(np.abs(m.coef_) > 1e-8))
    filas.append({"modelo": nombre,
                  "mse_test": round(mean_squared_error(yte, pred), 5),
                  "r2_test": round(r2_score(yte, pred), 4),
                  "n_variables": n_vars})
reg_comparacion = pd.DataFrame(filas)
print(reg_comparacion.to_string(index=False))


📖 **Lectura de negocio -- que dice la tabla.**
- **OLS** con 122 variables no colapsa (R2 de prueba ~0.62), pero es el que más
  sobreajusta: su brecha entrenamiento-prueba (~0.09) es la más ancha de los tres. Es
  el costo, con números, del Paso 1.
- **RIDGE** reduce esa brecha a ~0.07: con los 122 coeficientes contraídos hacia cero, el R2 de
  prueba sube a **~0.63**.
- **LASSO** la reduce aun más, a ~0.02, y logra el mejor R2 de prueba, **~0.64**, con
  **solo 27 de las 122** variables: iguala o mejora a RIDGE (la diferencia de MSE esta
  **dentro del ruido** de la validación cruzada) con un modelo mucho más simple.

💡 **Titular honesto para la aseguradora.** Los tres modelos predicen en un rango
parecido en datos nuevos, pero OLS desaprovecha más precisión por sobreajuste; RIDGE
y LASSO lo recortan, y LASSO además selecciona: la misma prima estimada con 27
indicadores en vez de 122, de menor costo de mantenimiento y más sencillo de explicar a un
regulador.

**Convención del curso.** Todo resultado de modelo se vuelca a
`resultados/E03_resultados.xlsx` y las **figuras se generan al leer ese Excel**, nunca
desde el objeto en memoria. Se aplica desde este primer resultado y se repite en cada
paso que produzca una figura.


In [ ]:
# Exportar la tabla comparativa (crea el Excel; los pasos siguientes agregan hojas).
reg_comparacion.to_excel(XLSX, sheet_name="reg_comparacion", index=False)
reg_leida = pd.read_excel(XLSX, sheet_name="reg_comparacion")
print("Hoja 'reg_comparacion' escrita y releida desde", XLSX)


In [ ]:
# Figura R1: error de prueba (MSE) por modelo, al leer el Excel (escala
# lineal: con datos bien procesados los tres modelos quedan en un rango
# parecido, no hay un derrumbe que forzar a escala log).
fig, ax = plt.subplots(figsize=(5.8, 3.9))
bars = ax.bar(reg_leida["modelo"], reg_leida["mse_test"], color=[UPC_GRAY, UPC_INK, UPC_RED])
ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=9)
ax.set_ylabel("MSE de prueba"); ax.set_ylim(0, reg_leida["mse_test"].max()*1.25)
ax.set_title("Error en datos NUEVOS: los tres modelos quedan cerca, RIDGE y LASSO ganan algo")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "res_reg_mse.png"), bbox_inches="tight"); plt.show()

In [ ]:
# Figura R2: número de variables que usa cada modelo, al leer el Excel
# (LASSO selecciona).
fig, ax = plt.subplots(figsize=(5.8, 3.9))
bars = ax.bar(reg_leida["modelo"], reg_leida["n_variables"], color=[UPC_GRAY, UPC_INK, UPC_RED])
ax.bar_label(bars, fmt="%d", padding=3)
ax.set_ylabel("Número de variables usadas"); ax.set_ylim(0, reg_leida["n_variables"].max()*1.15)
ax.set_title("El LASSO usa muchas menos variables (misma predicción)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "res_reg_nvars.png"), bbox_inches="tight"); plt.show()


**Punto de control -- completar:** Si RIDGE y LASSO predicen prácticamente igual, ¿por qué la aseguradora preferiría igual el LASSO?

> _______________________________________________________________________


## Paso 6 de 11 -- ¿Qué variables conserva el LASSO, y qué se puede (y no se puede) afirmar de ellas?

*(lectura breve, 6 min)* El LASSO del Paso 5 (`alpha` elegido por CV) selecciono 27 de
122 variables. Falta ver **cuales** son y que lectura de negocio admiten -y ponerle
precio a la parsimonia.


In [ ]:
# Que variables CONSERVA el LASSO (las que "sobreviven" a la penalización).
coef_lasso = pd.Series(lasso_cv.coef_, index=X.columns)
vivas = coef_lasso[coef_lasso.abs() > 1e-8]
vivas = vivas.reindex(vivas.abs().sort_values(ascending=False).index)
reg_lasso_vars = (vivas.head(12).rename("coef_estandarizado").round(4)
                  .reset_index().rename(columns={"index": "variable"}))
print(f"El LASSO conserva {len(vivas)} de {X.shape[1]} variables. Las 12 más influyentes:")
print(reg_lasso_vars.to_string(index=False))


📖 **Lectura de las variables seleccionadas.** El LASSO se queda con un conjunto
**interpretable** de predictores: el porcentaje de nacimientos fuera de matrimonio
(`PctIlleg`), la estructura familiar (`PctKids2Par`, `MalePctDivorce`), la composición
demográfica (`racepctblack`, `racePctWhite`), la urbanización (`pctUrban`) o el número de
personas sin hogar (`NumStreet`). Con los predictores estandarizados, los coeficientes
son comparables entre sí: el de mayor valor absoluto es el más influyente.

⚠️ **Tres avisos honestos.** (1) Que una variable **sobreviva no prueba que "cause"** la
tasa de crimen: LASSO selecciona por capacidad **predictiva**, no causal. (2) La
selección **no es única ni perfectamente estable**: cambia algo con la muestra y con
`alpha`. (3) Que una variable quede **fuera no la vuelve irrelevante** en el mundo real;
solo dice que, dadas las demás, no aportaba lo suficiente.


In [ ]:
# Exportar las variables seleccionadas (registro, sin figura: convención del curso).
with pd.ExcelWriter(XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
    reg_lasso_vars.to_excel(w, sheet_name="reg_lasso_vars", index=False)
print("Hoja 'reg_lasso_vars' escrita en", XLSX)


In [ ]:
# Ponerle precio a la parsimonia: variables descartadas x costo anual por
# variable (licencia de datos + integración/ETL + monitoreo).
costo_anual_por_variable = 5000  # USD, supuesto ilustrativo -sustituir por el del caso propio-
descartadas = X.shape[1] - len(vivas)
ahorro_anual = descartadas * costo_anual_por_variable
print(f"Variables descartadas: {descartadas} de {X.shape[1]}")
print(f"Ahorro estimado: {descartadas} x US$ {costo_anual_por_variable:,} = US$ {ahorro_anual:,} / año")


💡 **Cierre del Acto 2.** El ahorro de US$ 475,000 (con el supuesto de US$ 5,000/variable/año) al año **solo** cuenta si la
precisión fuera de muestra se mantiene (Paso 5 ya lo confirmo: LASSO iguala a RIDGE). La
regla que se lleva la aseguradora: **contraer** (RIDGE) estabiliza sin descartar variables;
**seleccionar** (LASSO) además abarata la operación. Con demasiadas variables
correlacionadas, regularizar es lo que vuelve usable el modelo.


**Punto de control -- completar:** Si un colega dice "las 27 variables que quedaron son las que CAUSAN el crimen", ¿qué se le respondería con los tres avisos honestos?

> _______________________________________________________________________


---
## Acto 3 -- Comprimir: PCA (Min 134-203)

**El caso de negocio (el mismo de los Actos 1 y 2).** La aseguradora de riesgos
patrimoniales del Acto 1 no solo quiere un modelo que estime con precisión la tasa de crimen: sus
analistas de riesgo también necesitan **visualizar** de un vistazo, en un mapa de dos
ejes, en que se parecen y en que difieren las comunidades de su cartera, sin mirar las
122 variables de a pares (inviable a escala). El **Análisis de Componentes Principales
(PCA)** resume esas variables en unas pocas **componentes** nuevas -combinaciones de
las originales- ordenadas por la cantidad de información (varianza) que capturan.

**Un solo dataset, dos preguntas de negocio.** Los Pasos 7-11 reutilizan exactamente
los mismos datos ya preparados en el Paso 1 -`Xtr`/`Xte` (imputados) y `Ztr`/`Zte`
(estandarizados), con la misma partición de entrenamiento y prueba-: no se carga ni se
prepara nada nuevo. Antes se uso LASSO para **seleccionar** que variables importan;
ahora se usa PCA para **comprimir** las 122 en unos pocos ejes y **visualizar** la
cartera.

**La idea, sin fórmulas.** El PCA busca la dirección donde los datos más **varian**
(PC1), luego la siguiente dirección independiente (PC2), y así. Quedarse con las
primeras componentes conserva buena parte de la información con muchas menos
dimensiones.


## Paso 7 de 11 -- ¿Por qué estandarizar antes de aplicar PCA, si los datos ya vienen en la misma escala [0, 1]?

*(lectura breve, 7 min)* Las columnas de `communities.csv` ya vienen normalizadas a la
escala **[0, 1]** por quien construyó el dataset (0 = mínimo observado entre las comunidades,
1 = máximo). Es tentador asumir que, si todas comparten rango, ya se puede aplicar PCA
sin más preparación. **No es así:** estar en el mismo **rango** no es lo mismo que
tener la misma **varianza**. La celda siguiente lo muestra con un dato concreto:
`pctUrban` (porcentaje de población urbana) tiene, en el entrenamiento, la **varianza
más alta de las 122 columnas** -muy por encima de la mediana de las demás-. El PCA
persigue **varianza**, así que sin estandarizar, `pctUrban` dominaría el primer eje no
porque sea el indicador más relevante del riesgo de crimen, sino porque su escala
interna concentra más dispersión que las demás.

**Estandarizar (z-score):** cada variable pasa a tener promedio 0 y varianza 1:

$$z = \frac{x-\mu}{\sigma}$$

Es exactamente el mismo `escalador` que ya se ajustó en el Paso 1
(`StandardScaler.fit(Xtr)`, celda `c1_d`): `Ztr` y `Zte` ya están listos, no hace
falta volver a ajustarlo. Con los datos estandarizados, el PCA resuelve un problema de
autovalores sobre la matriz de correlación $S$: cada autovector $v$ es una dirección
(una componente) y su autovalor $\lambda$ es la varianza que capta en esa dirección.

**De dónde viene el PCA (Pearson, 1901).** El antecedente del método no buscaba
comprimir datos ni visualizarlos: buscaba la recta -o el plano- de **mejor ajuste** a una
nube de puntos cuando **ninguna** variable merece el rango de respuesta y todas se
midieron con error. Pearson reparó eso con una decisión que él mismo calificó de
arbitraria pero que resultó fértil: minimizar las distancias **perpendiculares** a la
recta en vez de las verticales, con lo que la solución deja de depender de qué variable
recibió el nombre de respuesta y pasa a ser **única**. De esa elección derivan dos hábitos
que este cuaderno repite sin discutirlos: la recta pasa por el **centroide** de la nube
-por eso el PCA centra los datos antes de nada- y la solución depende solo de medias,
desviaciones típicas y **correlaciones** -por eso la escala entra en el problema, y por
eso el PCA sobre datos crudos y sobre datos estandarizados no dan el mismo resultado,
justo lo que muestra la celda siguiente-. Precisión al citarlo: el artículo de 1901
aporta el problema y su solución geométrica; el **porcentaje de varianza explicada** es
la formalización estadística posterior de Hotelling (1933). Desarrollo completo en
`REPLICACION_PAPER.md` de S06, Sección 0; aquí solo se condensa.

La celda siguiente compara la varianza de `pctUrban` con el resto, y luego ejecuta el PCA
**dos veces** -sobre `Xtr` (imputado, sin estandarizar) y sobre `Ztr` (estandarizado)-
para exhibir el efecto de la escala.


In [ ]:
# Aunque las 122 columnas comparten el rango [0, 1], sus VARIANZAS no son iguales.
varianzas = pd.DataFrame(Xtr, columns=Xtr_df.columns).var()
print(f"Varianza de pctUrban:                        {varianzas['pctUrban']:.3f}")
print(f"Varianza mediana de las otras 121 columnas:  {varianzas.drop('pctUrban').median():.3f}")
print(f"Puesto de pctUrban por varianza (1 = la más alta de las 122): "
      f"{int(varianzas.rank(ascending=False)['pctUrban'])}")


In [ ]:
# PCA sobre los datos CRUDOS: Xtr ya viene imputado (Paso 1) pero SIN estandarizar.
pca_crudo = PCA().fit(Xtr)
vr_c = pca_crudo.explained_variance_ratio_
print(f"PC1 sin estandarizar: {vr_c[0]*100:.1f}% de la varianza")
print(f"PC2 sin estandarizar: {vr_c[1]*100:.1f}% de la varianza")


In [ ]:
# PCA sobre los datos ESTANDARIZADOS: se reutiliza Ztr, ya calculado en el Paso 1
# (el mismo escalador, ajustado SOLO con el entrenamiento).
pca_std = PCA().fit(Ztr)
vr_s = pca_std.explained_variance_ratio_
cum_s = np.cumsum(vr_s)
print(f"PC1 estandarizado: {vr_s[0]*100:.1f}% de la varianza")
print(f"PC2 estandarizado: {vr_s[1]*100:.1f}% de la varianza")


In [ ]:
# Tabla de la convención: crudo vs. estandarizado.
n80 = int(np.searchsorted(cum_s, 0.80) + 1)
n90 = int(np.searchsorted(cum_s, 0.90) + 1)
pca_convencion = pd.DataFrame({
    "convención": ["Crudo (sin estandarizar)", "Estandarizado"],
    "PC1_%": [round(vr_c[0]*100, 2), round(vr_s[0]*100, 2)],
    "PC2_%": [round(vr_c[1]*100, 2), round(vr_s[1]*100, 2)],
    "comps_para_90%": [int(np.searchsorted(np.cumsum(vr_c), 0.90)+1), n90],
})
print(pca_convencion.to_string(index=False))


📖 **Lectura -- por qué "misma escala" no es "misma varianza".**
- **Crudo (`Xtr`, sin estandarizar):** PC1 explica **25.2%** de la varianza, PC2
  **18.4%**. No es una cifra tan alarmante como el 99.8% que produce una variable en
  cientos frente a otras en decimales (el ejemplo clasico de este sesgo), pero no deja de ser un **artefacto de escala**: `pctUrban`, con la mayor varianza de las 122
  columnas, pesa de más solo por su dispersión interna, no porque sea el indicador más
  relevante del riesgo de crimen.
- **Estandarizado (`Ztr`):** el reparto cambia -PC1 **20.7%**, PC2 **14.4%**, acumulado
  **35.1%**- y hacen falta **16 de las 122 componentes** para llegar al 80%, y **30**
  para el 90%. Ninguna variable domina por su escala; el reparto refleja la correlación
  real entre los indicadores socioeconómicos.

⚠️ **Riesgo (el mismo de la regularización, en otra forma).** "Estar en la misma escala"
no es lo mismo que "tener la misma varianza". Se estandariza SIEMPRE antes de PCA,
incluso cuando los datos ya parecen comparables a simple vista.


In [ ]:
# Exportar la convención (agrega hojas de PCA al Excel; ya trae las hojas de RIDGE/LASSO).
with pd.ExcelWriter(XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
    pca_convencion.to_excel(w, sheet_name="pca_convencion", index=False)
conv_leida = pd.read_excel(XLSX, sheet_name="pca_convencion")
print("Hoja 'pca_convencion' escrita y releida desde", XLSX)


In [ ]:
# Figura R3: PC1 crudo vs. estandarizado, al leer el Excel.
fig, ax = plt.subplots(figsize=(5.8, 3.9))
bars = ax.bar(conv_leida["convención"], conv_leida["PC1_%"], color=[UPC_GRAY, UPC_RED])
ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set_ylabel("Varianza explicada por PC1 (%)"); ax.set_ylim(0, 35)
ax.set_title("Sin estandarizar, PC1 sube por la varianza de pctUrban")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "res_pca_convencion.png"), bbox_inches="tight"); plt.show()


**Punto de control -- completar:** Un colega observa que las 122 columnas de `communities.csv` ya están en el rango [0, 1] y decide saltarse la estandarización antes del PCA. ¿Qué se le respondería, con la varianza de `pctUrban` como argumento?

> _______________________________________________________________________


## Paso 8 de 11 -- ¿Cuántas componentes conviene retener?

*(lectura breve, 5 min)* Cada componente trae una **varianza explicada** (que fracción
de la varianza total retiene). Al sumarlas se obtiene la **varianza acumulada**, que
responde "con k componentes, cuánto se conserva". Fórmula (con datos estandarizados,
$\sum\lambda = 122$, el número de variables de Communities and Crime):

$$\text{var. explicada}_k = \frac{\lambda_k}{\lambda_1+\dots+\lambda_{122}}$$

Así se decide cuantas retener: por la acumulada frente a un **umbral** (80-95% es lo
típico), no de forma arbitraria ni por una sola componente. El umbral no es único: cambia según el
objetivo (visualizar vs. modelar), como muestra la celda siguiente.


In [ ]:
# Varianza explicada y acumulada (estandarizado) para elegir cuantas componentes.
pca_varianza = pd.DataFrame({
    "componente": [f"PC{i+1}" for i in range(len(vr_s))],
    "var_explicada_%": (vr_s*100).round(2),
    "acumulada_%": (cum_s*100).round(2),
})
print(pca_varianza.head(8).to_string(index=False))


In [ ]:
print(f"-> Para VISUALIZAR en 2D bastan 2 componentes (PC1+PC2 = {cum_s[1]*100:.1f}% de la información).")
print(f"-> Para retener el 80% de la información harían falta {n80} de las 122 componentes.")
print(f"-> Para retener el 90% de la información harían falta {n90} de las 122 componentes.")


📖 **Lectura.** Con 2 componentes se conserva apenas el **35.1%** (suficiente para
*ver* en 2D, Paso 10, pero lejos de "casi toda la información"); para retener el **80%**
hacen falta **16** de las 122 componentes, y para el **90%**, **30**. Con 122 variables
originales fuertemente correlacionadas entre sí, comprimir a 16-30 componentes no deja de ser un recorte sustancial (de 122 a 16 es ~87% menos ejes).

**El número se ata al uso, no es una cifra fija.** 2 componentes alcanzan para comunicar un
mapa a un comité (Paso 10); 16-30 alcanzan para modelar sin perder casi nada de la
información original. No hay una respuesta única: depende de si el objetivo es *ver* o
*modelar*.


In [ ]:
with pd.ExcelWriter(XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
    pca_varianza.to_excel(w, sheet_name="pca_varianza", index=False)
var_leida = pd.read_excel(XLSX, sheet_name="pca_varianza")
print("Hoja 'pca_varianza' escrita y releida desde", XLSX)


In [ ]:
# Figura R4: scree / varianza acumulada (primeras 20 de 122), al leer el Excel.
top = var_leida.head(20)
fig, ax = plt.subplots(figsize=(7.4, 4.2))
bars = ax.bar(top["componente"], top["var_explicada_%"], color=UPC_GRAY, label="Var. explicada")
ax.plot(top["componente"], top["acumulada_%"], color=UPC_RED, marker="o", label="Acumulada")
ax.axhline(90, color=UPC_INK, linestyle="--", linewidth=1)
ax.axhline(80, color=UPC_INK, linestyle=":", linewidth=1)
ax.text(0.2, 91.5, "90%", color=UPC_INK, fontsize=9)
ax.text(0.2, 81.5, "80%", color=UPC_INK, fontsize=9)
ax.set_ylabel("Varianza (%)"); ax.set_title("PCA de Communities and Crime: primeras 20 de 122 componentes")
ax.tick_params(axis="x", labelrotation=90, labelsize=7)
ax.legend(loc="center right")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "res_pca_scree.png"), bbox_inches="tight"); plt.show()


**Punto de control -- completar:** Si el objetivo fuera *modelar* con al menos el 90% de la información original conservada, ¿cuántas de las 122 componentes se retienen? ¿Y si el objetivo fuera solo *visualizar* la cartera en un plano?

> _______________________________________________________________________


## Paso 9 de 11 -- ¿Cómo se le pone nombre a una componente?

*(lectura breve, 7 min)* Un eje llamado "PC1" no dice nada en una reunion: hace falta
ponerle nombre y respaldarlo con un número. La **carga** (*loading*) de una variable en
una componente -lo que permite nombrar el eje- es la coordenada del autovector escalada
por $\sqrt{\lambda}$; coincide con la correlación entre la variable y la componente:

$$\ell_{jk} = v_{jk}\,\sqrt{\lambda_k}$$

⚠️ **`pca.components_` NO es la carga.** Lo que devuelve `scikit-learn` en
`pca.components_` es el **autovector unitario** ($v_{jk}$), **no** la carga -le falta el
factor $\sqrt{\lambda}$-. Se miran las cargas de mayor **valor absoluto** para nombrar
el eje; el **signo** solo indica dirección (el eje puede venir volteado), nunca calidad.


In [ ]:
# Autovector unitario (pca.components_): la DIRECCIÓN del eje, no la carga.
autovec = pd.DataFrame(pca_std.components_[:2].T, index=X.columns, columns=["PC1", "PC2"])
pca_cargas = autovec.round(3).reset_index().rename(columns={"index": "variable"})
print("PC1 - autovector unitario (4 variables con mayor valor absoluto):")
print(autovec["PC1"].reindex(autovec["PC1"].abs().sort_values(ascending=False).index).head(4).round(3).to_string())


In [ ]:
# Carga de correlación = autovector * raiz(autovalor de la matriz de correlación).
# autovalor_k = ratio_k * p (p=122 variables; los autovalores suman p).
lam_corr = vr_s * X.shape[1]
cargas = pd.DataFrame((pca_std.components_[:2].T) * np.sqrt(lam_corr[:2]),
                      index=X.columns, columns=["PC1", "PC2"])
pca_cargas_corr = cargas.round(3).reset_index().rename(columns={"index": "variable"})
print("PC1 - carga de correlación (8 variables con mayor valor absoluto):")
print(cargas["PC1"].reindex(cargas["PC1"].abs().sort_values(ascending=False).index).head(8).round(3).to_string())
print("\nPC2 - carga de correlación (8 variables con mayor valor absoluto):")
print(cargas["PC2"].reindex(cargas["PC2"].abs().sort_values(ascending=False).index).head(8).round(3).to_string())
print(f"\n[control] medFamInc en PC1: autovector={autovec.loc['medFamInc','PC1']:.3f}"
      f"  ->  carga={cargas.loc['medFamInc','PC1']:.3f}"
      f"  (= {autovec.loc['medFamInc','PC1']:.3f} * sqrt({lam_corr[0]:.3f}))")


📖 **Lectura de negocio.** Las cargas convierten un eje abstracto en una historia.

**PC1** carga fuerte y positivo en `medFamInc` (0.90), `medIncome` (0.89),
`PctKids2Par` (0.87) y `pctWInvInc` (0.86), y fuerte y negativo en `PctPopUnderPov`
(-0.87) y `pctWPubAsst` (-0.82): ingreso familiar, familias con dos padres e ingreso
por inversión suben el eje; pobreza y asistencia pública lo bajan. Se nombra
**"bienestar socioeconómico"**.

**PC2** carga fuerte en `PctRecImmig8` (0.87), `PctRecImmig10` (0.87),
`PctForeignBorn` (0.85) y `PctRecentImmig` (0.85), con signo opuesto en
`PctSpeakEnglOnly` (-0.75): mide la proporción de población inmigrante reciente y
nacida en el extranjero. Se nombra **"composición demográfica reciente"**. Es una
lectura **puramente descriptiva** de como el censo mide la composición de una
comunidad -no una asociación con el crimen-: la correlación de PC2 con la tasa de
crimen es tenue (~0.24), muy por debajo de la de PC1 (~-0.64, ver Paso 10). El eje que
realmente separa comunidades por riesgo es PC1, no PC2.

⚠️ **Riesgo: una componente NO es una variable original.** PC1 no es "el ingreso
familiar": es una **mezcla** de decenas de variables. Se gana compresión y una vista
2D, pero se pierde el nombre original de cada variable -por eso, cuando el negocio
necesita decir "subió el ingreso familiar promedio de la zona", conviene LASSO
(conserva variables); cuando necesita "ver el mapa de la cartera", PCA.


In [ ]:
with pd.ExcelWriter(XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
    pca_cargas.to_excel(w, sheet_name="pca_cargas", index=False)            # autovector unitario
    pca_cargas_corr.to_excel(w, sheet_name="pca_cargas_corr", index=False)  # carga de correlación
print("Hojas 'pca_cargas' y 'pca_cargas_corr' escritas en", XLSX)


**Punto de control -- completar:** Un analista dice "el autovector de `medFamInc` en PC1 es 0.18, ese es su peso real". ¿Qué número debería citar en su lugar, y por qué?

> _______________________________________________________________________


## Paso 10 de 11 -- ¿Qué muestra (y qué no) un mapa en 2D?

*(lectura breve, 4 min)* Con solo **PC1 y PC2** (35.1% de la información, Paso 8) se
puede dibujar un plano con las comunidades de entrenamiento, coloreadas por **tercil de
tasa de crimen** (bajo / medio / alto, calculado sobre `ytr`), y ver si el "bienestar
socioeconómico" (PC1) separa a las comunidades de más y de menos riesgo -sin haberle
dicho nada al PCA sobre el crimen (es una técnica NO supervisada: PC1 y PC2 se
calcularon solo a partir de los 122 predictores).


In [ ]:
# Coordenadas de cada comunidad de entrenamiento en PC1-PC2, más su tercil de crimen
# (calculado sobre ytr, solo para colorear el gráfico; el PCA no lo uso para nada).
scores = pca_std.transform(Ztr)[:, :2]
terciles = pd.qcut(ytr, 3, labels=["bajo", "medio", "alto"])
pca_scores = pd.DataFrame({"pc1": scores[:, 0].round(4), "pc2": scores[:, 1].round(4),
                           "tercil_crimen": terciles.astype(str)})
pca_scores.head()


In [ ]:
# PC1 promedio por tercil de crimen: separación limpia y monótona.
resumen_tercil = (pca_scores.groupby("tercil_crimen", observed=True)["pc1"]
                  .agg(["mean", "count"]).reindex(["bajo", "medio", "alto"]))
print(resumen_tercil.round(3).to_string())


In [ ]:
with pd.ExcelWriter(XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
    pca_scores.to_excel(w, sheet_name="pca_scores", index=False)
    resumen_tercil.reset_index().to_excel(w, sheet_name="pca_tercil", index=False)
sco_leido = pd.read_excel(XLSX, sheet_name="pca_scores")
print("Hojas 'pca_scores' y 'pca_tercil' escritas y releidas desde", XLSX)


In [ ]:
# Figura R5: comunidades de entrenamiento en 2D (PC1-PC2), coloreadas por tercil de
# tasa de crimen, al leer el Excel.
orden = ["bajo", "medio", "alto"]
colores = {"bajo": UPC_GREEN, "medio": UPC_GRAY, "alto": UPC_RED}
fig, ax = plt.subplots(figsize=(6.2, 4.6))
for t in orden:
    s = sco_leido[sco_leido["tercil_crimen"] == t]
    ax.scatter(s["pc1"], s["pc2"], s=30, alpha=0.7, color=colores[t], label=f"Crimen {t}")
ax.set_xlabel("PC1 (bienestar socioeconómico)"); ax.set_ylabel("PC2 (composición demográfica reciente)")
ax.set_title("Communities and Crime en 2D: PC1 separa por riesgo de crimen")
ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(FIG, "res_pca_scatter.png"), bbox_inches="tight"); plt.show()


📖 **Lectura del mapa 2D.** Con apenas 2 de 122 dimensiones (35.1% de la
información), las comunidades de **bajo** riesgo (PC1 promedio +3.48), **medio**
(+0.28) y **alto** (-4.20) ya aparecen separadas de forma nítida y monótona a lo largo
de PC1: a mayor "bienestar socioeconómico", menor tasa de crimen. Eso es lo que hace
valiosa a PCA en negocio: permite **ver** en 2D la estructura de una cartera de miles
de comunidades, clientes o zonas para detectar grupos, atípicos o patrones que después
se estudian con detalle o se llevan a un modelo.

⚠️ **Riesgo: cuanta varianza se deja fuera.** El mapa usa PC1+PC2 = **35.1%** de la
información; el otro **64.9%** no esta en la vista. Basta para **explorar** y comunicar
una tendencia clara, pero **no** para fijar una prima de forma precisa: dos comunidades
cercanas en el plano pueden diferir en las dimensiones ocultas (para eso hacen falta las
30 componentes del 90%, Paso 8, o directamente el modelo LASSO del Acto 2).


**Punto de control -- completar:** Dos comunidades aparecen muy cerca en el plano PC1-PC2. ¿Se puede concluir con seguridad que tienen un riesgo de crimen casi idéntico? ¿Por qué?

> _______________________________________________________________________


---
## Acto 4 -- Decidir (Min 203-240)
## Paso 11 de 11 -- ¿Cuándo conviene LASSO y cuándo PCA?

*(lectura breve, 6 min)* Ante **demasiadas variables**, hay dos caminos, y la elección
depende del objetivo de la aseguradora:

| Objetivo de negocio | Técnica | Que entrega |
|---|---|---|
| Quedarse con las **variables que importan** (y poder nombrarlas) | **LASSO** | Un modelo **ralo**, interpretable y de bajo costo operativo |
| Estabilizar un modelo que conserva todas las variables | **RIDGE** | Coeficientes contraídos hacia cero, más fiables ante correlación |
| **Comprimir / visualizar** muchas variables correlacionadas | **PCA** | Pocas **componentes** nuevas; permite ver en 2D |

**Diferencia clave:** el **LASSO conserva variables originales** (descarta unas y
mantiene otras, con su significado intacto: `medFamInc`, `PctKids2Par`...); el **PCA
crea variables nuevas** (combinaciones de las originales, como el "bienestar
socioeconómico" del Paso 9) que resumen, pero ya no son "el ingreso familiar" o "el
porcentaje de pobreza". Por eso LASSO se usa cuando la aseguradora necesita **explicar
con las variables de siempre** -por ejemplo, ante un regulador que exige saber que
indicador fija la prima-, y PCA cuando necesita **comprimir o visualizar** su cartera
completa.

**Valor en dinero, para cerrar el "so what" (un solo caso, dos decisiones).** El LASSO
del Acto 2 descarta 95 de 122 variables: con el supuesto de costo del Paso 6, unos
**US$ 475 000/año** de ahorro de recolección. La compresión por PCA tiene una lectura
monetaria distinta: comprimir de 122 a 30 componentes (90% de la información) recorta
~75% del espacio; a escala industrial (p. ej. embeddings de un sistema de búsqueda o de
detección de fraude con esa compresión) ese tipo de ahorro se multiplica en
almacenamiento y computo -el mismo principio, otra escala.

**Recomendación para la aseguradora, con 122 variables sobre la mesa:** (1) estandarizar
siempre; (2) si el objetivo es fijar la prima con un modelo interpretable ante el
regulador, LASSO (penalización por validación cruzada, error en datos de prueba); (3) si
el objetivo es explorar o visualizar la cartera completa, PCA (número de componentes por
varianza acumulada, ejes nombrados con las cargas); (4) juzgar siempre por el error o la
separación fuera de muestra.


**Punto de control -- completar:** Para un caso propio con muchas variables correlacionadas, ¿cuál técnica se usaría primero (LASSO o PCA) y por qué?

> _______________________________________________________________________


---
## Cierre, entregable y control corto (Min 218-240)

**Recapitulación del arco.** Se vio **el problema** (muchas variables correlacionadas
producen sobreajuste, Paso 1), se **contrajeron** los coeficientes con RIDGE y LASSO -que además selecciona-
(Pasos 2-6), se **comprimió** el mismo dataset con PCA -varianza explicada y mapa 2D
sobre la cartera de comunidades- (Pasos 7-10) y se **decidió** qué técnica conviene
según el objetivo (Paso 11).

### Entregable de la sesión
Ver `evaluación/entregable.docx`: aplicar **PCA** a un caso **nuevo** -una productora de
vinos que necesita clasificar sus cultivares (dataset *Wine*, UCI)-, que este cuaderno
no trabajó, para evaluar si el método (estandarizar, elegir componentes, nombrar ejes
con sus cargas, visualizar en 2D) se transfiere a datos distintos de los de clase. Se
califica con rúbrica **vigesimal (total = 20)**. La celda «Punto de control - completar:» de cada
paso se apoya en `plantillas/plantilla_comparacion_modelos.docx` (Pasos 1-6) y
`plantillas/guia_componentes_pca.docx` (Pasos 7-11).

### Control corto (Min 225-240)
Cierra la sesión un control corto individual de 8 preguntas (2.5 puntos c/u = 20),
sobre las nueve ideas recorridas en los 11 pasos.

### Para profundizar
El docente comparte en clase casos recientes (2025-2026) sobre selección de variables
e interpretabilidad en riesgo de crédito y de seguros, y sobre PCA/embeddings para
comprimir y monitorear datos de alta dimensión, con enlaces verificados.

### Bibliografía (lecturas de la sesión)
- James, Witten, Hastie & Tibshirani. *An Introduction to Statistical Learning* (ISLR), cap. 6.1-6.2 (selección y regularización) y 12.2 (componentes principales). [statlearning.com](https://www.statlearning.com/)
- Provost, F. & Fawcett, T. (2013). *Data Science for Business*, cap. 4-5 (sobreajuste y complejidad del modelo). O'Reilly.

**Próxima sesión (E4 -- Segmentación).** Agrupará clientes justamente sobre las
variables que aquí seleccionó el LASSO y las componentes que aquí comprimió el PCA.

---
*Cuaderno de la Sesión EPE E3 - Herramientas de Ciencias de Datos - UPC. Dataset:
Communities and Crime (UCI), usado en las dos técnicas de la sesión (RIDGE/LASSO y
PCA). El dataset Wine (UCI) se reserva para el entregable, como caso de transferencia.
Todas las cifras provienen de la ejecución de este cuaderno y de
`resultados/E03_resultados.xlsx`.*
